# IPC 코드 비교: gnn_266_phase3 vs 전체 USPTO

**목적**: `gnn_266_phase3.ipynb`에서 학습된 IPC 임베딩(218개)이 전체 USPTO 특허의 IPC 코드 공간을 얼마나 커버하는지 확인.

- **266 IPC**: 소송 1,331개 특허에서 추출한 section+ipc_class+subclass 코드 → 218개 (모두 A~H section, 4글자)
- **Full IPC**: 전체 USPTO g_ipc_at_issue.tsv — **266과 동일 조건으로 필터**: section A~H + 길이 4글자

> **주의**: 전체 IPC TSV엔 숫자·소문자 section(0~9, a~z, ?) 등 오염 데이터가 포함됨.
> 266 소송 특허는 자연스럽게 정상 코드만 갖고 있으므로, 전체 데이터도 동일 기준(A~H, 4글자)으로 필터해야 공정한 비교 가능.

→ 266 학습 모델 가중치를 새 특허에 재학습 없이 적용할 때, IPC lookup이 실패하는 비율을 파악.

In [7]:
# Cell 1 — Imports
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

ROOT     = Path('../../..').resolve()   # MANTF-master
SRC      = ROOT / 'src'
DATA     = ROOT / 'data' / 'samples'
IPC_TSV  = SRC / 'g_ipc_at_issue.tsv'
BOTH_CSV = DATA / 'both_side_patents.csv'

print('ROOT    :', ROOT)
print('IPC_TSV :', IPC_TSV.exists(), f'  ({IPC_TSV.stat().st_size/1e9:.1f} GB)')
print('BOTH_CSV:', BOTH_CSV.exists())

ROOT    : /home2/jmbyeon/developments/tf_litigation/MANTF-master
IPC_TSV : True   (1.9 GB)
BOTH_CSV: True


In [8]:
# Cell 2 — 266 소송 특허 목록 (gnn_266_phase3.ipynb 동일 로직)
def parse_pipe(s):
    if pd.isna(s): return []
    return [x.strip() for x in str(s).split('|') if x.strip()]

raw      = pd.read_csv(BOTH_CSV)
pat_rows = []
for _, r in raw.iterrows():
    for pat in parse_pipe(r['P_patents']):
        pat_rows.append(pat)
    for pat in parse_pipe(r['D_patents']):
        pat_rows.append(pat)

ALL_266_PATENTS = set(pat_rows)
print(f'266 소송 특허 수: {len(ALL_266_PATENTS):,}')

266 소송 특허 수: 1,331


In [9]:
# Cell 3 — 전체 IPC TSV 로드 + 266과 동일 조건으로 필터
# 266 데이터는 자연스럽게 section A~H, 4글자만 나옴
# 전체 데이터엔 오염(숫자·소문자 section, 길이 3/5/6 등)이 있으므로 동일 기준 적용
VALID_SECTIONS = set('ABCDEFGH')

print('전체 IPC TSV 로드 중... (약 2,550만 행)')
ipc_raw = pd.read_csv(
    IPC_TSV, sep='\t',
    usecols=['patent_id', 'section', 'ipc_class', 'subclass'],
    dtype=str
)
ipc_raw['patent_id'] = ipc_raw['patent_id'].str.strip()
ipc_raw['ipc_code']  = (
    ipc_raw['section'].fillna('') +
    ipc_raw['ipc_class'].fillna('') +
    ipc_raw['subclass'].fillna('')
).str.strip()

# 266과 동일 조건: section A~H, 길이 4
ipc_raw = ipc_raw[
    ipc_raw['section'].isin(VALID_SECTIONS) &
    (ipc_raw['ipc_code'].str.len() == 4)
]

print(f'필터 후 행 수  : {len(ipc_raw):,}')
print(f'전체 특허 수   : {ipc_raw["patent_id"].nunique():,}')
print(f'전체 IPC 코드 수 (A~H, 4글자): {ipc_raw["ipc_code"].nunique():,}')

전체 IPC TSV 로드 중... (약 2,550만 행)
필터 후 행 수  : 18,332,807
전체 특허 수   : 7,385,800
전체 IPC 코드 수 (A~H, 4글자): 4,133


In [10]:
# Cell 4 — 266 소송 특허의 IPC 코드 (gnn_266_phase3 학습에 쓰인 것)
ipc_266 = ipc_raw[ipc_raw['patent_id'].isin(ALL_266_PATENTS)].copy()
ipc_266 = ipc_266.drop_duplicates(['patent_id', 'ipc_code'])

ipc_codes_266  = set(ipc_266['ipc_code'].unique())

print(f'266 소송 특허 중 IPC 있는 특허: {ipc_266["patent_id"].nunique():,} / {len(ALL_266_PATENTS):,}')
print(f'266 IPC 코드 수 (gnn_266_phase3 학습): {len(ipc_codes_266):,}')

df_266 = pd.DataFrame(sorted(ipc_codes_266), columns=['ipc_code'])
df_266['source'] = '266_phase3'
print('\n266 IPC 코드 목록 (상위 20개):')
print(df_266.head(20).to_string(index=False))

266 소송 특허 중 IPC 있는 특허: 1,281 / 1,331
266 IPC 코드 수 (gnn_266_phase3 학습): 209

266 IPC 코드 목록 (상위 20개):
ipc_code     source
    A01D 266_phase3
    A01G 266_phase3
    A01H 266_phase3
    A01K 266_phase3
    A01M 266_phase3
    A01N 266_phase3
    A16K 266_phase3
    A23B 266_phase3
    A23F 266_phase3
    A23K 266_phase3
    A23L 266_phase3
    A41C 266_phase3
    A41D 266_phase3
    A43B 266_phase3
    A43C 266_phase3
    A45F 266_phase3
    A47B 266_phase3
    A47C 266_phase3
    A47F 266_phase3
    A47G 266_phase3


In [11]:
# Cell 5 — 전체 USPTO IPC 코드 집합
ipc_codes_full = set(ipc_raw['ipc_code'].unique())

df_full = pd.DataFrame(sorted(ipc_codes_full), columns=['ipc_code'])
df_full['source'] = 'full_uspto'

print(f'전체 USPTO IPC 코드 수: {len(ipc_codes_full):,}')
print('\n전체 IPC 코드 목록 (상위 20개):')
print(df_full.head(20).to_string(index=False))

전체 USPTO IPC 코드 수: 4,133

전체 IPC 코드 목록 (상위 20개):
ipc_code     source
    A00C full_uspto
    A00K full_uspto
    A00N full_uspto
    A01A full_uspto
    A01B full_uspto
    A01C full_uspto
    A01D full_uspto
    A01E full_uspto
    A01F full_uspto
    A01G full_uspto
    A01H full_uspto
    A01I full_uspto
    A01J full_uspto
    A01K full_uspto
    A01L full_uspto
    A01M full_uspto
    A01N full_uspto
    A01O full_uspto
    A01P full_uspto
    A01Q full_uspto


In [12]:
# Cell 6 — 집합 비교
in_both    = ipc_codes_266 & ipc_codes_full    # 266에도 있고 full에도 있음
only_266   = ipc_codes_266 - ipc_codes_full    # 266에만 있음 (이론상 없어야 함)
only_full  = ipc_codes_full - ipc_codes_266    # full에만 있음 → 모델이 본 적 없는 IPC

coverage = len(in_both) / len(ipc_codes_full) * 100

print('=' * 55)
print(f'  266 phase3 IPC 코드 수        : {len(ipc_codes_266):>8,}')
print(f'  전체 USPTO IPC 코드 수        : {len(ipc_codes_full):>8,}')
print('-' * 55)
print(f'  교집합 (both)                 : {len(in_both):>8,}')
print(f'  266에만 있음 (only_266)       : {len(only_266):>8,}')
print(f'  전체에만 있음 (미학습 IPC)    : {len(only_full):>8,}')
print('-' * 55)
print(f'  266 모델의 전체 IPC 커버율    : {coverage:>7.2f}%')
print('=' * 55)

if only_266:
    print(f'\n[주의] 266에만 있는 IPC (비정상): {sorted(only_266)}')

  266 phase3 IPC 코드 수        :      209
  전체 USPTO IPC 코드 수        :    4,133
-------------------------------------------------------
  교집합 (both)                 :      209
  266에만 있음 (only_266)       :        0
  전체에만 있음 (미학습 IPC)    :    3,924
-------------------------------------------------------
  266 모델의 전체 IPC 커버율    :    5.06%


In [13]:
# Cell 7 — Section별 코드 수 분포 (A~H 기준, 이미 필터됨)
sec_266  = set(c[0] for c in ipc_codes_266  if c)
sec_full = set(c[0] for c in ipc_codes_full if c)

print('Section 분포 (A~H):')
print(f'  266  sections: {sorted(sec_266)}')
print(f'  Full sections: {sorted(sec_full)}')

# Section별 코드 수
print('\nSection별 고유 IPC 코드 수:')
print(f'{"Section":>8} {"Full USPTO":>12} {"266 phase3":>12} {"Coverage":>10}')
print('-' * 46)
for sec in sorted(sec_full):
    full_n = sum(1 for c in ipc_codes_full if c.startswith(sec))
    n266   = sum(1 for c in ipc_codes_266  if c.startswith(sec))
    cov    = n266 / full_n * 100 if full_n else 0
    print(f'{sec:>8} {full_n:>12,} {n266:>12,} {cov:>9.1f}%')

Section 분포 (A~H):
  266  sections: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
  Full sections: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

Section별 고유 IPC 코드 수:
 Section   Full USPTO   266 phase3   Coverage
----------------------------------------------
       A          587           38       6.5%
       B          879           35       4.0%
       C          536           23       4.3%
       D          274            3       1.1%
       E          287           12       4.2%
       F          569           21       3.7%
       G          538           37       6.9%
       H          463           40       8.6%


In [14]:
# Cell 8 — 미포함 IPC가 전체 특허 중 얼마나 되는지
# 즉, 새 특허가 들어왔을 때 IPC lookup 실패할 확률
unknown_mask  = ~ipc_raw['ipc_code'].isin(ipc_codes_266)
n_total_rows  = len(ipc_raw)
n_unknown_rows = unknown_mask.sum()

pats_with_unknown = ipc_raw[unknown_mask]['patent_id'].nunique()
pats_total        = ipc_raw['patent_id'].nunique()
pats_all_covered  = ipc_raw.groupby('patent_id')['ipc_code'].apply(
    lambda codes: all(c in ipc_codes_266 for c in codes)
).sum()

print('새 특허 적용 시 IPC lookup 실패 위험도:')
print(f'  전체 IPC 행 중 미학습 코드 포함: {n_unknown_rows:,} / {n_total_rows:,}  ({n_unknown_rows/n_total_rows*100:.1f}%)')
print(f'  미학습 IPC 보유 특허 수        : {pats_with_unknown:,} / {pats_total:,}  ({pats_with_unknown/pats_total*100:.1f}%)')
print(f'  모든 IPC가 커버되는 특허 수    : {pats_all_covered:,} / {pats_total:,}  ({pats_all_covered/pats_total*100:.1f}%)')

새 특허 적용 시 IPC lookup 실패 위험도:
  전체 IPC 행 중 미학습 코드 포함: 4,209,443 / 18,332,807  (23.0%)
  미학습 IPC 보유 특허 수        : 2,126,767 / 7,385,800  (28.8%)
  모든 IPC가 커버되는 특허 수    : 5,259,033 / 7,385,800  (71.2%)


In [ ]:
# Cell 9 — Section별 266 커버율 막대 그래프
section_stats = []
for sec in sorted(sec_full):
    full_in_sec = {c for c in ipc_codes_full if c.startswith(sec)}
    in_266      = len(full_in_sec & ipc_codes_266)
    total       = len(full_in_sec)
    section_stats.append({'section': sec, 'in_266': in_266, 'total': total,
                          'coverage': in_266 / total * 100 if total else 0})

df_sec = pd.DataFrame(section_stats)
print(df_sec.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Section별 IPC 코드 수 (266 vs full)
x   = np.arange(len(df_sec))
w   = 0.35
ax1 = axes[0]
ax1.bar(x - w/2, df_sec['total'], w, label='Full USPTO', color='#90CAF9')
ax1.bar(x + w/2, df_sec['in_266'], w, label='266 phase3', color='#1565C0')
ax1.set_xticks(x); ax1.set_xticklabels(df_sec['section'])
ax1.set_ylabel('IPC code count')
ax1.set_title('Section별 IPC 코드 수\n(Full USPTO vs 266 phase3)')
ax1.legend()

# Panel 2: Section별 커버율
ax2 = axes[1]
colors = ['#E53935' if v < 50 else '#FF9800' if v < 80 else '#43A047'
          for v in df_sec['coverage']]
bars = ax2.bar(df_sec['section'], df_sec['coverage'], color=colors)
for bar, v in zip(bars, df_sec['coverage']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax2.axhline(100, color='gray', linestyle='--', linewidth=0.8)
ax2.set_ylim(0, 115)
ax2.set_ylabel('Coverage (%)')
ax2.set_title('266 phase3 모델의 Section별 IPC 커버율\n(녹색≥80%, 주황50~80%, 빨강<50%)')

plt.suptitle('IPC 코드 비교: gnn_266_phase3 vs 전체 USPTO', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('ipc_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ipc_coverage.png')

In [ ]:
# Cell 10 — 미학습 IPC 코드 목록 (전체에만 있는 것)
df_only_full = pd.DataFrame(sorted(only_full), columns=['ipc_code'])
df_only_full['section']  = df_only_full['ipc_code'].str[0]
df_only_full['sec_class'] = df_only_full['ipc_code'].str[:3]

print(f'미학습 IPC 코드 총 {len(df_only_full):,}개')
print()
print('Section별 분포:')
print(df_only_full['section'].value_counts().to_string())
print()
print('상위 20개 미학습 IPC:')
# 각 미학습 코드가 전체 특허에서 몇 번 등장하는지
freq = ipc_raw[ipc_raw['ipc_code'].isin(only_full)]['ipc_code'].value_counts()
df_miss_freq = df_only_full.merge(
    freq.rename('freq').reset_index().rename(columns={'index': 'ipc_code'}),
    on='ipc_code', how='left'
).sort_values('freq', ascending=False)
print(df_miss_freq.head(20).to_string(index=False))

In [ ]:
# Cell 11 — 최종 요약
print('=' * 60)
print('결론: gnn_266_phase3 모델을 새 특허에 재학습 없이 적용 시')
print('=' * 60)
print(f'  학습된 IPC 코드 수    : {len(ipc_codes_266):,} / {len(ipc_codes_full):,} ({len(ipc_codes_266)/len(ipc_codes_full)*100:.2f}%)')
print(f'  미학습 IPC 코드 수    : {len(only_full):,}개 → ipc_emb lookup 불가')
print(f'  IPC 완전 커버 특허    : {pats_all_covered:,} / {pats_total:,} ({pats_all_covered/pats_total*100:.1f}%)')
print(f'  IPC 미커버 특허       : {pats_with_unknown:,} / {pats_total:,} ({pats_with_unknown/pats_total*100:.1f}%)')
print()
print('대안:')
print('  1) 미학습 IPC는 zero-vector or mean-vector로 대체')
print('  2) phase3_patent_faiss 모델 사용 (IPC 노드 없음 → 제약 없음)')
print('  3) 전체 USPTO IPC 코드로 ipc_emb 확장 후 fine-tuning')